# CourtListener Training Data Builder

pulls public bankruptcy filings from courtlistener and turns them into training data for both stages of the project:

- **time entries**: real attorney billing narratives, used to ground our synthetic data
- **fee examiner reports**: what examiners flag and why, used to build out the compliance checker

**before you run anything:**
1. put your courtlistener api token in a `.env` file in this folder: `COURTLISTENER_TOKEN=your_token_here` (sign in -> profile -> your api token)
2. make sure `data/raw/raw_scrape.json.gz` exists.
3. leave `RUN_SCRAPE = False` unless you're adding new cases. a full scrape takes hours

**what comes out** (all in `data/final/`):

| file | what it is |
|---|---|
| `time_entries_all.csv.gz` | every parsed entry (open with `pd.read_csv`, it handles the .gz), flagged by confidence and type. our audit trail |
| `time_entries_clean_balanced.csv` | clean attorney entries, capped per case. **use this for synthetic data** |
| `examiner_reports.json.gz` | deduped fee examiner reports (full text) |
| `examiner_paragraphs.csv` | report paragraphs tagged by violation category. **start the compliance read-through here** |
| `examiner_flagged_entries.csv` | individual entries quoted in reports (only shows up if exhibits got scraped) |

**a note on keyword lists:** the lists in here are either things every bankruptcy filing shares (court stamps, numbered paragraphs, UTBMS task codes, the U.S. Trustee guideline categories) or search terms. nothing is written around a specific firm or case, so it should hold up as we add cases. if something breaks on a new case, look for the general pattern before adding a one-off fix

## Setup
imports and the settings you'll actually touch

In [1]:
import os
import re
import gzip
import json
import time
from collections import Counter

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
# flip to True only when you've added new cases to cases.csv
RUN_SCRAPE = False

# max entries per case in the balanced set, so one huge case doesn't drown out the rest
MAX_ENTRIES_PER_CASE = 3000

RAW_DIR = "data/raw"
DOC_CACHE_DIR = "data/raw/docs"
INTERMEDIATE_DIR = "data/intermediate"
FINAL_DIR = "data/final"

RAW_PATH = f"{RAW_DIR}/raw_scrape.json.gz"
WITH_TEXT_PATH = f"{INTERMEDIATE_DIR}/docs_with_text.json.gz"
SECTIONS_PATH = f"{INTERMEDIATE_DIR}/time_entry_sections.json.gz"

for folder in [RAW_DIR, DOC_CACHE_DIR, INTERMEDIATE_DIR, FINAL_DIR]:
    os.makedirs(folder, exist_ok=True)

### File Helpers
everything big gets saved gzipped so it stays under github's 100MB limit. these handle both `.json` and `.json.gz`

In [3]:
def load_json(path):
    """loads a json file, gzipped or not"""
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        return json.load(f)


def save_json(data, path):
    """saves to json, gzips it if the path ends in .gz"""
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "wt", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)

### Boilerplate Cleanup
every filing has stuff that repeats at the top or bottom of each page: court stamps, firm letterhead, invoice headers. instead of listing those out per firm, we find lines that repeat at the edges of most pages (recap text marks page breaks with `\x0c`) and strip them. `COURT_STAMP` is a backup for docs without page breaks, it's the standard stamp every bankruptcy court puts on filings

In [4]:
COURT_STAMP = re.compile(r"^.*(Doc \d+\s+Filed|Page \d+ of \d+|Pg \d+ of \d+|Main Document).*$", re.M | re.I)


def normalize_line(line):
    """lowercases a line and swaps digits for # so 'Page 3 of 40' and 'Page 4 of 40' match"""
    return re.sub(r"\d+", "#", " ".join(line.split())).lower()


def find_page_boilerplate(text, edge_lines=4, min_share=0.5):
    """finds lines that repeat in the top/bottom few lines of at least half the pages"""
    pages = [p for p in text.split("\x0c") if p.strip()]
    if len(pages) < 3:
        return set()

    counts = Counter()
    for page in pages:
        lines = [l for l in page.split("\n") if l.strip()]
        counts.update({normalize_line(l) for l in lines[:edge_lines] + lines[-edge_lines:]})

    threshold = max(3, min_share * len(pages))
    return {line for line, n in counts.items() if n >= threshold}

In [5]:
def strip_boilerplate(text):
    """removes repeated page headers/footers and court stamps from a document"""
    boilerplate = find_page_boilerplate(text)
    if boilerplate:
        text = "\n".join(l for l in text.split("\n") if normalize_line(l) not in boilerplate)
    return COURT_STAMP.sub("", text)

## Cases and Search Settings
cases live in `cases.csv` (columns: `name`, `docket_number`, `court`). add new ones there as you find them. cases with an appointed fee examiner are the most useful

In [6]:
cases_df = pd.read_csv("cases.csv")
CASES = cases_df.to_dict("records")
print(f"{len(CASES)} cases in cases.csv")

# what we search each docket for
KEYWORDS = [
    "fee examiner",
    "interim compensation",
    "interim fee application",
    "final fee application",
    "monthly fee statement",
    "fee committee",
    "objection to fee",
    "compensation and reimbursement",
]

# only these get a second api call to pull full text (keeps request count down)
HIGH_VALUE_TERMS = ["fee examiner", "examiner's report", "monthly fee statement", "interim compensation"]

6 cases in cases.csv


In [7]:
TOKEN = os.environ.get("COURTLISTENER_TOKEN")
HEADERS = {"Authorization": f"Token {TOKEN}"}
API_ROOT = "https://www.courtlistener.com/api/rest/v4"

REQUEST_DELAY = 3.5         # seconds between calls, keeps us under 20/min
DAILY_REQUEST_LIMIT = 5000  # safety stop
request_count = 0

if RUN_SCRAPE and not TOKEN:
    raise ValueError("no COURTLISTENER_TOKEN found, add it to your .env file")

## API Helpers
`safe_get` paces itself and retries when we get rate limited. `get_doc_cached` saves every document we fetch to `data/raw/docs/` so reruns never re-download the same thing

In [8]:
def safe_get(url, params=None, retries=3, backoff=2):
    """grabs a url with pacing + retries. returns the json, or None if it keeps failing"""
    global request_count

    if request_count >= DAILY_REQUEST_LIMIT:
        print("hit the safety stop, raise DAILY_REQUEST_LIMIT if you need more")
        return None

    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
            request_count += 1
            time.sleep(REQUEST_DELAY)

            if resp.status_code == 429:
                wait = backoff * (2 ** attempt)
                print(f"rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue
            if 400 <= resp.status_code < 500:
                # 404, 403, etc. retrying won't help
                print(f"got {resp.status_code} for {url}, skipping")
                return None

            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as e:
            print(f"request failed ({e}), retry {attempt + 1}/{retries}")
            time.sleep(backoff)

    print(f"giving up on {url}")
    return None

In [9]:
def get_doc_cached(doc_id):
    """fetches one recap document, using the local cache if we already have it"""
    path = f"{DOC_CACHE_DIR}/{doc_id}.json"
    if os.path.exists(path):
        return load_json(path)

    data = safe_get(f"{API_ROOT}/recap-documents/{doc_id}/")
    if data:
        save_json(data, path)
    return data

## Scraping Functions
search a docket for fee-related filings (filtered server-side so we're not pulling everything), flatten the results into a list of documents, then grab full text for the high-value ones

In [10]:
def search_docket(docket_number, court):
    """searches one docket for fee-related filings. returns (results, incomplete)"""
    keyword_clause = " OR ".join(f'"{kw}"' for kw in KEYWORDS)
    query = f'docketNumber:"{docket_number}" AND court_id:{court} AND ({keyword_clause})'

    url = f"{API_ROOT}/search/"
    params = {"q": query, "type": "rd", "page_size": 100}
    results, incomplete = [], False

    while url:
        data = safe_get(url, params=params)
        if not data:
            incomplete = True
            break
        results.extend(data.get("results", []))
        url = data.get("next")
        params = None  # the "next" url already has the params baked in

    return results, incomplete

In [11]:
def flatten_documents(search_results):
    """pulls documents out of search results. attachments (like "Exhibit B") get
    their parent's description tacked on, so we still know what they belong to"""
    docs = []
    for r in search_results:
        nested = r.get("recap_documents")
        if not nested:
            docs.append(r)
            continue
        parent_desc = r.get("description", "") or ""
        for d in nested:
            d["description"] = f"{parent_desc} | {d.get('description', '') or ''}"
            docs.append(d)
    return docs


def is_high_value(description):
    """True if this doc is worth an extra api call to get its text"""
    d = (description or "").lower()
    return any(term in d for term in HIGH_VALUE_TERMS)

In [12]:
def fetch_text(doc):
    """returns (plain_text, absolute_url, failed) for one doc, fetching full text only if it's worth it"""
    plain_text = doc.get("plain_text")
    absolute_url = doc.get("absolute_url")
    worth_fetching = doc.get("is_available") and not plain_text and is_high_value(doc.get("description"))

    if worth_fetching and doc.get("id"):
        full = get_doc_cached(doc["id"])
        if not full:
            return plain_text, absolute_url, True
        plain_text = full.get("plain_text")
        absolute_url = full.get("absolute_url")

    return plain_text, absolute_url, False

In [13]:
def process_case(case):
    """scrapes one case. returns (rows, incomplete)"""
    name, docket, court = case["name"], case["docket_number"], case["court"]
    print(f"=== {name} ({docket}, {court}) ===")

    results, incomplete = search_docket(docket, court)
    documents = flatten_documents(results)
    print(f"  {len(documents)} fee-related docs found" + (" [SEARCH INCOMPLETE]" if incomplete else ""))
    if not documents:
        print("  WARNING: 0 docs, double check the docket_number format in cases.csv")

    rows = []
    for doc in documents:
        plain_text, absolute_url, failed = fetch_text(doc)
        incomplete = incomplete or failed
        rows.append({
            "case_name": name, "docket_number": docket, "court": court,
            "document_id": doc.get("id"), "description": doc.get("description", ""),
            "is_available": doc.get("is_available", False),
            "plain_text": plain_text, "absolute_url": absolute_url,
        })

    with_text = sum(1 for r in rows if r["plain_text"])
    print(f"  {with_text} of {len(rows)} had free text\n")
    return rows, incomplete

## Run the Scrape
skips any case that's already fully scraped and saves after every case, so if it crashes halfway you don't lose anything. only runs when `RUN_SCRAPE = True`

In [14]:
if not RUN_SCRAPE:
    assert os.path.exists(RAW_PATH), f"{RAW_PATH} is missing. grab it, or set RUN_SCRAPE = True"
    print("RUN_SCRAPE is False, using the existing raw file")
else:
    all_results = load_json(RAW_PATH) if os.path.exists(RAW_PATH) else []
    done = {(r["docket_number"], r["court"]) for r in all_results if r.get("_case_complete")}

    for case in CASES:
        key = (case["docket_number"], case["court"])
        if key in done:
            print(f"skipping {case['name']}, already scraped")
            continue

        all_results = [r for r in all_results if (r["docket_number"], r["court"]) != key]
        rows, incomplete = process_case(case)
        for r in rows:
            r["_case_complete"] = not incomplete
        all_results.extend(rows)
        save_json(all_results, RAW_PATH)

    print(f"done. {len(all_results)} documents total, {request_count} requests used")

RUN_SCRAPE is False, using the existing raw file


## Load and Keep Documents With Text
everything below this point works off the saved raw file, so it never hits the api

In [15]:
all_results = load_json(RAW_PATH)
with_text = [r for r in all_results if r["plain_text"]]
save_json(with_text, WITH_TEXT_PATH)

print(f"{len(with_text)} documents with text, out of {len(all_results)} total")

3170 documents with text, out of 8269 total


## Violation Categories
keywords for tagging what examiners flag, based on the U.S. Trustee fee guideline categories. shared by Part 1 and Part 3, so tweak them here. if you see examiners using a new phrase for the same thing, add it to the matching list

In [16]:
CATEGORIES = {
    "vague": ["vague", "insufficient detail", "lacks specificity", "lack of detail", "not described with sufficient"],
    "block_billing": ["block bill", "lumping", "lumped"],
    "staffing": ["staffing", "overstaffing", "multiple attendees", "transitory", "transient"],
    "administrative": ["clerical", "administrative task", "secretarial"],
    "fee_app_prep": ["fee application preparation", "retention application"],
    "travel": ["non-working travel", "travel time"],
}

# Part 1: Fee Examiner Reports

### Filter to Real Reports
the search pulls in a lot of noise (certificates of service, notices, the FTX *investigation* examiner, etc). we only want actual fee examiner reports. the noise lists are standard filing types, not anything case specific

In [17]:
NOISE_PREFIXES = [
    "affidavit", "certificate", "certification", "application to employ",
    "application for pro hac vice", "declaration", "order authorizing",
    "notice of appearance", "monthly fee statement",
]
NOISE_ATTACHMENTS = ["certificate of service", "proposed order", "notice"]


def is_real_report(description):
    """True if the description looks like an actual fee examiner report"""
    parent, _, attachment = (description or "").lower().partition(" | ")
    parent = parent.strip()

    if "fee examiner" not in parent and "fee committee" not in parent:
        return False
    if any(parent.startswith(p) for p in NOISE_PREFIXES):
        return False
    if any(w in attachment for w in NOISE_ATTACHMENTS):
        return False

    # "examiner" and "report" should show up near the start, not buried in something like
    # "objection to the fee examiner's report"
    examiner_pos, report_pos = parent.find("examiner"), parent.find("report")
    return 0 <= examiner_pos < 30 and 0 <= report_pos < 100

### Dedupe Reports
some reports got refiled as "corrected" versions. we keep the corrected one, otherwise the newest

In [18]:
def report_key(r):
    """groups a report with its corrected/refiled versions (but not with its own attachments)"""
    parent, _, attachment = r["description"].partition(" | ")

    def norm(s):
        s = s.lower().replace("corrected ", "")
        return " ".join(re.sub(r"[^a-z ]+", " ", s).split())

    return (r["case_name"], norm(parent)[:120], norm(attachment))


def dedupe_reports(reports):
    """keeps one report per key: the corrected version if there is one, otherwise the newest"""
    best = {}
    for r in reports:
        k = report_key(r)
        keep = best.get(k)
        r_fixed = "corrected" in r["description"].lower()
        keep_fixed = keep is not None and "corrected" in keep["description"].lower()

        if keep is None or (r_fixed and not keep_fixed) or (r_fixed == keep_fixed and r["document_id"] > keep["document_id"]):
            best[k] = r
    return list(best.values())

In [19]:
reports = dedupe_reports([r for r in with_text if is_real_report(r["description"])])
save_json(reports, f"{FINAL_DIR}/examiner_reports.json.gz")

for r in reports:
    print(f"[{r['case_name']}] {r['description'][:80]}")
print(f"\n{len(reports)} unique examiner reports")

[Seadrill Limited] Notice Fee Examiner's Report Regarding Review of the First Interim Fee Applicati
[Seadrill Limited] Notice FEE EXAMINERS REPORT REGARDING: (I) REVIEW OF THE FINAL FEE APPLICATIONS 
[BlockFi Inc.] Document re: Fee Examiners Corrected Consolidated Final Report Regarding Final F
[BlockFi Inc.] Document re: Fee Examiner's Consolidated Final Report Regarding Final Fee Applic
[BlockFi Inc.] Document re: Fee Examiners Consolidated Final Report Regarding Final Fee Applica
[BlockFi Inc.] Document re: Fee Examiner's Consolidated Final Report Regarding Second Interim F
[BlockFi Inc.] Document re: Fee Examiner's Consolidated Final Report Regarding Second Interim F
[BlockFi Inc.] Document re: Fee Examiners Consolidated Final Report Regarding First Interim Fee
[FTX Trading Ltd.] Examiner's Report / Fee Examiner's Summary Report on Fee Review Process and Firs
[FTX Trading Ltd.] Examiner's Report /Fee Examiner's Summary Report on Fee Review Process and Seven
[FTX Trading Ltd.] Exami

### Rebuild Paragraphs
court filings are double spaced, so there's a blank line after *every* line and we can't just split on blank lines. instead we use two conventions court filings follow:
- paragraphs are numbered (`45.  Proper time keeping...`)
- sections get lettered or roman numeral headings (`D.  Lumping; Failure to Bill...`)

headings only get picked up in some reports (others use different styles), so `section` will often be blank. that's fine, the paragraphs still come out whole. tables and long lists don't have numbers, so anything too long gets split at sentence breaks

In [20]:
PARA_START = re.compile(r"^\s*\d{1,3}\.\s+\S")                          # "45.  Proper time keeping..."
HEADING = re.compile(r"^\s*(?:[A-Z]|[IVX]{1,5})\.\s+([A-Z].{2,100})$")   # "D.  Lumping; Failure to..."
TOC_LEADER = re.compile(r"\.{4,}")                                      # table of contents dot leaders
SENTENCE_END = re.compile(r"(?<=[.!?])[”\"]?\s+(?=[A-Z(“])")


def chunk_long(paragraph, max_chars=2000):
    """splits a too-long paragraph (usually a table or list) into sentence-sized chunks"""
    if len(paragraph) <= max_chars:
        return [paragraph]
    chunks, current = [], ""
    for sentence in SENTENCE_END.split(paragraph):
        if current and len(current) + len(sentence) > max_chars:
            chunks.append(current.strip())
            current = ""
        current += " " + sentence
    if current.strip():
        chunks.append(current.strip())
    return chunks

In [21]:
def split_report_paragraphs(text):
    """rebuilds real paragraphs from a report. returns a list of (section heading, paragraph text)"""
    lines = [l for l in strip_boilerplate(text).split("\n") if l.strip()]
    paragraphs, current, section = [], [], None

    def flush():
        if current:
            joined = " ".join(" ".join(current).split())
            paragraphs.extend((section, chunk) for chunk in chunk_long(joined))

    for line in lines:
        heading = HEADING.match(line)
        if heading and not TOC_LEADER.search(line) and not line.rstrip().endswith((",", ";")):
            flush()
            current = []
            section = " ".join(heading.group(1).split())
        elif PARA_START.match(line):
            flush()
            current = [line]
        elif not TOC_LEADER.search(line):
            current.append(line)

    flush()
    return paragraphs

### Paragraph Bank
keeps the paragraphs that talk about a violation, tagged by category. this is the starting point for the compliance read-through. `is_guidance` marks "should / must / requires" language, which is usually the "here's what a good entry looks like" stuff. sort by `section` to see the examiner's own groupings when they're there

In [22]:
MONEY = re.compile(r"\$[\d,]+(?:\.\d{2})?")
GUIDANCE = re.compile(r"\b(should|must|require[sd]?|guidelines? provide|sufficient detail)\b", re.I)


def tag_paragraphs(report):
    """returns one row per paragraph that mentions a violation category, with the section it's under"""
    rows = []
    for section, p in split_report_paragraphs(report["plain_text"]):
        searchable = f"{section or ''} {p}".lower()
        hits = [c for c, kws in CATEGORIES.items() if any(k in searchable for k in kws)]
        if hits and len(p) > 80:
            rows.append({
                "case_name": report["case_name"],
                "document_id": report["document_id"],
                "section": section,
                "categories": ",".join(hits),
                "is_guidance": bool(GUIDANCE.search(p)),
                "amounts": ",".join(MONEY.findall(p)),
                "text": p,
                "absolute_url": report.get("absolute_url"),
            })
    return rows

In [23]:
paragraphs = pd.DataFrame([row for r in reports for row in tag_paragraphs(r)])
paragraphs.to_csv(f"{FINAL_DIR}/examiner_paragraphs.csv", index=False)

print(f"{len(paragraphs)} paragraphs tagged, {paragraphs['is_guidance'].sum()} with guidance language\n")
print(paragraphs["categories"].str.split(",").explode().value_counts())

274 paragraphs tagged, 83 with guidance language

categories
staffing          113
vague              97
fee_app_prep       96
block_billing      82
travel             12
administrative     11
Name: count, dtype: int64


### Report Health Check
one row per report. a really low `paragraphs` count or a tiny `median_len` means a new format the splitter doesn't handle, so open that report before trusting its rows. a blank `pct_with_section` is normal

In [24]:
report_health = pd.DataFrame([
    {
        "case_name": r["case_name"],
        "document_id": r["document_id"],
        "paragraphs": len(ps),
        "median_len": int(pd.Series([len(p) for _, p in ps]).median()) if ps else 0,
        "pct_with_section": round(100 * sum(s is not None for s, _ in ps) / len(ps)) if ps else 0,
    }
    for r in reports
    for ps in [split_report_paragraphs(r["plain_text"])]
])
report_health.sort_values("median_len")

,case_name,document_id,paragraphs,median_len,pct_with_section
13,FTX Trading Ltd.,488992062,13,249,0
11,FTX Trading Ltd.,442973412,24,293,0
12,FTX Trading Ltd.,432002204,61,302,0
16,FTX Trading Ltd.,410879948,50,334,0
15,FTX Trading Ltd.,401246289,46,343,0
8,FTX Trading Ltd.,353413192,119,352,90
9,FTX Trading Ltd.,422333947,53,357,0
22,Celsius Network LLC,410541803,14,363,0
21,Celsius Network LLC,273242647,142,374,94
24,Celsius Network LLC,362590600,94,390,0


# Part 2: Time Entries

### Date Formats
entries always start with a date. covers `1/17/23`, `01-17-2023`, `2023-01-17`, and `Jan. 17, 2023`. the same pattern is used to find the entry section and to split it into entries

In [25]:
MONTH = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?"
DATE_REGEX = (
    r"(\d{1,2}[-/]\d{1,2}[-/]\d{2,4}"   # 1/17/23, 01-17-2023
    r"|\d{4}-\d{2}-\d{2}"               # 2023-01-17
    rf"|{MONTH}\s+\d{{1,2}},?\s+\d{{4}})"  # Jan. 17, 2023
)
DATE_PATTERN = re.compile(rf"^{DATE_REGEX}(?=\s|$)")
DATE_LINE = re.compile(rf"^\s*{DATE_REGEX}(?=\s|$)", re.M)

### Find the Time-Entry Section
fee statements bury the itemized entries in an exhibit. first we look for a section marker like "time detail" that's followed by actual date lines (so we skip table of contents mentions). if a doc doesn't use any of those markers, we fall back to the first big run of date lines. either way we stop before the expense section. `found_by` records which one worked so we can check it

In [26]:
TIME_DETAIL_MARKERS = ["time detail", "detailed time records", "time log", "daily time log"]
END_MARKERS = ["expense detail", "summary of expenses", "disbursement", "expenses incurred"]
MIN_DATE_LINES = 25  # fallback only kicks in for docs with a real run of entries, not a small summary table


def cut_at_expenses(section):
    """trims a section right before the expense part starts"""
    ends = [i for i in (section.lower().find(e) for e in END_MARKERS) if i > 0]
    return section[:min(ends)] if ends else section

In [27]:
def find_time_entry_section(text):
    """returns (section, found_by) for the itemized time entries, or (None, None) if there aren't any"""
    lower = text.lower()
    for marker in TIME_DETAIL_MARKERS:
        for m in re.finditer(re.escape(marker), lower):
            if DATE_LINE.search(text, m.end(), m.end() + 3000):
                return cut_at_expenses(text[m.start():]), "marker"

    date_lines = list(DATE_LINE.finditer(text))
    if len(date_lines) >= MIN_DATE_LINES:
        return cut_at_expenses(text[date_lines[0].start():]), "date_lines"

    return None, None

### Who Billed It
we only want entries written by attorneys. financial advisors and claims agents bill too, but their entries read totally differently. instead of listing firm names, we look at how the filing describes the applicant. pacer descriptions often say "filed by Financial Advisor ...", and fee statements describe themselves as "financial advisor to the Official Committee" and so on

heads up: almost every filing has a footnote mentioning the "debtors' claims and noticing agent", so we only count a role when it's followed by "to the debtors" / "for the committee" (that's how applicants describe themselves)

In [28]:
def extract_firm(description):
    """best guess at the billing firm from the filing description. returns None if it can't tell"""
    d = " ".join((description or "").split())
    m = re.search(
        r"(?:statement|application) of ([A-Z][\w&.,' ]+?(?:LLP|LLC|L\.L\.P\.|P\.C\.|P\.A\.|Inc\.?)(?![a-z]))",
        d, re.I,
    )
    if m:
        return m.group(1).strip(" ,")
    m = re.search(r"[Ff]iled by (?:Attorney |Other Professional )?([^.(|]+)", d)
    return m.group(1).strip() if m else None

In [29]:
ROLE_IN_DESCRIPTION = re.compile(
    r"filed by (financial advisor|claims and noticing agent|claims agent|accountant|consultant|auditor)", re.I)
ROLE_IN_TEXT = re.compile(
    r"(financial advis[eo]rs?|investment bank(?:ers?)?|accountants?|consultants?|"
    r"claims (?:and |& )?(?:noticing |solicitation )?agent|noticing agent)"
    r"\s+(?:to|for)\s+(?:the\s+)?(?:debtor|official|ad hoc|committee|chapter|fee examiner|examiner|unsecured)", re.I)


def classify_professional(description, text):
    """guesses who billed this from how the filing describes the applicant.
    returns 'attorney', 'advisor', or 'claims_agent'. defaults to attorney if nothing matches"""
    m = ROLE_IN_DESCRIPTION.search(description or "") or ROLE_IN_TEXT.search((text or "")[:5000])
    if not m:
        return "attorney"
    return "claims_agent" if "agent" in m.group(1).lower() else "advisor"

### Pull Out the Sections

In [30]:
report_ids = {r["document_id"] for r in reports}  # reports have summary tables with dates, skip them

sections = []
for r in with_text:
    if r["document_id"] in report_ids:
        continue
    text = strip_boilerplate(r["plain_text"])
    section, found_by = find_time_entry_section(text)
    if section:
        sections.append({
            "case_name": r["case_name"],
            "firm": extract_firm(r["description"]),
            "role": classify_professional(r["description"], text),
            "found_by": found_by,
            "document_id": r["document_id"],
            "description": r["description"],
            "raw_entry_section": section,
            "absolute_url": r.get("absolute_url"),
        })

save_json(sections, SECTIONS_PATH)
print(f"{len(sections)} documents have a time-entry section, out of {len(with_text)} with text\n")
print(pd.DataFrame(sections)[["role", "found_by"]].value_counts())

1346 documents have a time-entry section, out of 3170 with text

role          found_by  
attorney      date_lines    573
              marker        566
advisor       date_lines    126
              marker         79
claims_agent  date_lines      2
Name: count, dtype: int64


### Parsing Patterns
hours can look like `.4`, `0.40`, or `5.9`. task codes are UTBMS codes like `L320` or `A104` (the standard legal billing codes). table header rows that leak into narratives get stripped too

In [31]:
HOURS_PATTERN = re.compile(r"(?<![\d.,$])(\d{0,2}\.\d{1,2})(?![\d,])")
AMOUNT_PATTERN = re.compile(r"(?<![\w(])\$?\s*[\d,]*\.\d{1,2}(?![\w)])")  # leaves "(0.8)" splits alone
TASK_CODE = re.compile(r"\b([ABEL]\d{3})\b")

NOT_NAMES = {"date", "mon", "tue", "wed", "thu", "fri", "sat", "sun",
             "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"}
ADMIN_WORDS = {"invoice", "total", "matter", "page", "client", "balance", "remittance", "desc"}

# when an entry runs across a page break, the next page's table header can land in the middle of it
# ("...and Prime Date Tkpr Task Hours Value Trust regarding same"). every billing table has a header
# row like this, so we strip any run of 3+ header words, plus invoice numbers and page numbers
HEADER_WORDS = r"(?:Date|Atty|Attorney|Task|Tkpr|Timekeeper|Professional|Person|Name|Description|Services|Rendered|Narrative|Hours|Hrs|Rate|Amount|Value|Of)"
TABLE_HEADER = re.compile(rf"(?:\b{HEADER_WORDS}\b[\s:.]*){{3,}}", re.I)
INVOICE_BITS = re.compile(r"\bInvoice\s*(?:No\.?|#|Number)\s*:?\s*[\d-]+|\bPage:?\s*\d+(?:\s+of\s+\d+)?", re.I)

### Parsing Helpers
small checks the main parser leans on

In [32]:
def looks_like_a_name(text):
    """True if a column looks like a timekeeper (a name, initials, or "Last, First")"""
    if not text or len(text) > 40:
        return False
    lower = text.lower()
    if lower.strip(" .") in NOT_NAMES or any(w in lower for w in ADMIN_WORDS):
        return False
    if any(c.isdigit() for c in text):
        return False
    return bool(re.fullmatch(r"[A-Za-z.,'\-\s]+", text))


def clean_narrative(text):
    """strips dollar amounts, bracketed notes, table headers, and leading matter/task numbers (like '0004') out of a narrative"""
    text = AMOUNT_PATTERN.sub(" ", text)
    text = re.sub(r"\[.*?\]", "", text)
    text = TABLE_HEADER.sub(" ", text)
    text = INVOICE_BITS.sub(" ", text)
    text = " ".join(text.split()).strip()
    return re.sub(r"^\d{3,6}\s+", "", text)

In [33]:
def split_into_entry_blocks(text):
    """splits a section into one block per entry. a new entry starts at every line that begins with a date"""
    blocks, current = [], []
    for line in text.split("\n"):
        line = line.strip()
        if not line:
            continue
        if DATE_PATTERN.match(line):
            if current:
                blocks.append("\n".join(current))
            current = [line]
        elif current:
            current.append(line)
    if current:
        blocks.append("\n".join(current))
    return blocks

### The Entry Parser
splits on wide column gaps instead of matching name patterns, so it doesn't care about capitalization. anything that doesn't look confident keeps its `raw_block` instead of guessing, so nothing gets silently wrong as we add cases

In [34]:
def parse_entry_block(block):
    """turns one raw entry block into date / timekeeper / hours / task codes / narrative"""
    date_match = DATE_PATTERN.match(block)
    date = date_match.group(1) if date_match else None
    remainder = block[date_match.end():] if date_match else block

    task_codes = TASK_CODE.findall(remainder.split("\n")[0])
    remainder = TASK_CODE.sub(" ", remainder)

    columns = [c.strip() for c in re.split(r"\s{2,}", remainder) if c.strip()]
    timekeeper = columns[0] if columns else None
    rest = " ".join(columns[1:])

    numbers = [float(n) for n in HOURS_PATTERN.findall(rest.split("\n")[0])]
    hours = next((n for n in numbers if 0 < n <= 24), None)

    narrative = clean_narrative(rest)
    name_ok = looks_like_a_name(timekeeper)
    high_confidence = bool(date and name_ok and hours is not None and len(narrative) > 5)

    return {
        "date": date,
        "timekeeper": timekeeper if name_ok else None,
        "hours": hours,
        "task_codes": ",".join(task_codes),
        "narrative": narrative if high_confidence else None,
        "high_confidence": high_confidence,
        "raw_block": block,
    }

In [35]:
all_entries = []
for doc in sections:
    for block in split_into_entry_blocks(doc["raw_entry_section"]):
        parsed = parse_entry_block(block)
        parsed.update({
            "case_name": doc["case_name"],
            "firm": doc["firm"],
            "role": doc["role"],
            "found_by": doc["found_by"],
            "document_id": doc["document_id"],
            "description": doc["description"],
            "absolute_url": doc["absolute_url"],
        })
        all_entries.append(parsed)

entries = pd.DataFrame(all_entries)
print(f"{len(entries)} entries parsed, {entries['high_confidence'].mean():.0%} high confidence")

235520 entries parsed, 79% high confidence


### Double-Check Who Billed It
the role check above only works when a filing spells out its role, and a lot don't (in NJ the description just says "filed by [attorney] on behalf of M3 Partners"). so we also look at what the entries actually talk about. attorney entries are full of legal work (motions, hearings, drafting), advisor entries are full of numbers work (analysis, models, decks), and claims agent entries are about creditor inquiries and mailings

we score each *document*, not each entry, since one entry is too short to judge. we checked this against filings where we know who the applicant was: it caught ~92% of advisor entries and ~95% of claims agent entries while keeping ~93% of attorney entries. the attorney entries it drops are mostly fee examiner's counsel reviewing other firms' bills, which isn't typical billing anyway

In [36]:
LEGAL_WORDS = re.compile(
    r"\b(?:motions?|briefs?|pleadings?|draft\w*|revis\w*|hearing|court|objections?|stipulation|order|research|"
    r"depositions?|discovery|privilege|memo\w*|complaint|docket|counsel|settlement|declaration|agreement|"
    r"statute|judge|chambers|filing|file)\b", re.I)
ADVISOR_WORDS = re.compile(
    r"\b(?:analys[ie]s|analyz\w*|model\w*|deck|presentation|waterfall|forecast\w*|cash|budget|reconcil\w*|"
    r"database|data|slides?|valuation|spreadsheet|dashboard|metrics|MOR|balance sheet|financial)\b", re.I)
AGENT_WORDS = re.compile(
    r"\b(?:creditor inquir\w*|inquiries|call center|noticing|mailing|ballots?|solicitation|claims register|"
    r"claim forms?|service list|tabulat\w*)\b", re.I)

MIN_ENTRIES_TO_JUDGE = 5   # docs with fewer clean entries than this are too small to score
AGENT_SHARE = 0.25         # if a quarter of a doc's entries are claims agent work, it's a claims agent

In [37]:
def guess_role_from_narratives(entries):
    """scores each document by what its entries talk about. returns a Series of
    document_id -> 'attorney', 'advisor', 'claims_agent', or 'too_few'"""
    hc = entries[entries["high_confidence"]]
    shares = pd.DataFrame({
        "document_id": hc["document_id"],
        "legal": hc["narrative"].str.contains(LEGAL_WORDS),
        "advisor": hc["narrative"].str.contains(ADVISOR_WORDS),
        "agent": hc["narrative"].str.contains(AGENT_WORDS),
    }).groupby("document_id").agg(
        n=("legal", "size"), legal=("legal", "mean"), advisor=("advisor", "mean"), agent=("agent", "mean"))

    def pick(r):
        if r["n"] < MIN_ENTRIES_TO_JUDGE:
            return "too_few"
        if r["agent"] >= AGENT_SHARE:
            return "claims_agent"
        return "advisor" if r["advisor"] > r["legal"] else "attorney"

    return shares.apply(pick, axis=1)

In [38]:
entries["vocab_role"] = entries["document_id"].map(guess_role_from_narratives(entries)).fillna("too_few")

# trust the filing's own description first, then fall back to the vocab check
entries["final_role"] = np.where(
    entries["role"] != "attorney", entries["role"],
    np.where(entries["vocab_role"].isin(["advisor", "claims_agent"]), entries["vocab_role"], "attorney"),
)
print(pd.crosstab(entries["role"], entries["vocab_role"]))
print()
print(entries["final_role"].value_counts())

vocab_role    advisor  attorney  claims_agent  too_few
role                                                  
advisor         14240      2471             0     1246
attorney        43045    134750         11223    28151
claims_agent        0         0             0      394

final_role
attorney        162901
advisor          61002
claims_agent     11617
Name: count, dtype: int64


### Tag, Dedupe, and Balance
- **tag**: only attorney-billed entries count (using `final_role` from above). expense lines that slip into the time section get dropped too. we only call something an expense if the narrative is short and expense-y, so a real entry like "research on Westlaw re: preference claims" doesn't get thrown out
- **dedupe**: interim applications re-attach the monthly statements, so the same entry shows up more than once
- **balance**: cap each case at `MAX_ENTRIES_PER_CASE` so one big case doesn't take over the style

In [39]:
EXPENSE_WORDS = re.compile(r"charges?|photocopy|copies|postage|courier|filing fee|lexis|westlaw|pacer|meals?|airfare|hotel", re.I)
MAX_EXPENSE_WORDS = 8


def is_expense_line(narrative):
    """True for short lines like 'Telecommunication Charges' that are really expenses, not time"""
    if not isinstance(narrative, str):
        return False
    return len(narrative.split()) <= MAX_EXPENSE_WORDS and bool(EXPENSE_WORDS.search(narrative))


def tag_entry_type(df):
    """adds an entry_type column: 'attorney' for attorney-billed time, 'exclude' for everything else"""
    keep = (df["final_role"] == "attorney") & ~df["narrative"].apply(is_expense_line)
    df["entry_type"] = np.where(keep, "attorney", "exclude")
    return df

In [40]:
entries = tag_entry_type(entries)
entries = entries.drop_duplicates(["case_name", "date", "timekeeper", "hours", "raw_block"])
entries.to_csv(f"{FINAL_DIR}/time_entries_all.csv.gz", index=False)  # gzipped, it's too big for github otherwise

clean = entries[entries["high_confidence"] & (entries["entry_type"] == "attorney")]
balanced = clean.sample(frac=1, random_state=42).groupby("case_name").head(MAX_ENTRIES_PER_CASE)
balanced.to_csv(f"{FINAL_DIR}/time_entries_clean_balanced.csv", index=False)

print(f"{len(entries)} unique entries, {len(clean)} clean attorney entries, {len(balanced)} in the balanced set")

196313 unique entries, 98348 clean attorney entries, 12000 in the balanced set


# Part 3: Entries Quoted in Reports
some reports quote the exact entries they flagged, usually in an exhibit. these would be our only true entry-level labels, so they're worth grabbing. each one gets labeled with whichever category was mentioned most recently above it

heads up: with our current data this comes back empty, since the quoted entries live in exhibits that go to the examiner privately. it'll start filling in if a case files them publicly

In [41]:
def nearest_category(text, pos, window=1500):
    """returns the violation category mentioned closest above a position in the text"""
    context = text[max(0, pos - window):pos].lower()
    last_seen = {c: max(context.rfind(k) for k in kws) for c, kws in CATEGORIES.items()}
    last_seen = {c: i for c, i in last_seen.items() if i != -1}
    return max(last_seen, key=last_seen.get) if last_seen else None

In [42]:
flagged = []
for r in reports:
    text = strip_boilerplate(r["plain_text"])
    for block in split_into_entry_blocks(text):
        parsed = parse_entry_block(block)
        if not parsed["high_confidence"]:
            continue
        pos = text.find(block.split("\n")[0])
        parsed.update({
            "flag_category": nearest_category(text, pos),
            "case_name": r["case_name"],
            "report_id": r["document_id"],
        })
        flagged.append(parsed)

if flagged:
    flagged_df = pd.DataFrame(flagged)
    flagged_df.to_csv(f"{FINAL_DIR}/examiner_flagged_entries.csv", index=False)
    print(flagged_df["flag_category"].value_counts(dropna=False))
else:
    print("no quoted entries found (expected unless a case files its exhibits publicly)")

no quoted entries found (expected unless a case files its exhibits publicly)


# Quality Checks

### Health Check per Case
one row per case. a low `pct_high_conf` usually means a new format the parser hasn't seen, so open it up. cases with 0 `with_text` just don't have free documents in recap. `excluded` is entries dropped for being non-attorney or expenses

In [43]:
raw_df = pd.DataFrame(all_results)
health = raw_df.groupby("case_name").agg(
    docs=("document_id", "nunique"),
    with_text=("plain_text", lambda s: s.notna().sum()),
)
health = health.join([
    pd.DataFrame(sections).groupby("case_name").size().rename("entry_docs"),
    entries.groupby("case_name").agg(entries=("date", "size"), pct_high_conf=("high_confidence", "mean")),
    entries[entries["entry_type"] == "exclude"].groupby("case_name").size().rename("excluded"),
    clean.groupby("case_name").size().rename("clean_entries"),
    pd.DataFrame(reports).groupby("case_name").size().rename("reports"),
]).fillna(0)

health["pct_high_conf"] = (health["pct_high_conf"] * 100).round(0)
health.sort_values("pct_high_conf")

,docs,with_text,entry_docs,entries,pct_high_conf,excluded,clean_entries,reports
case_name,,,,,,,,
Seadrill Limited,128,2,0.0,0.0,0.0,0.0,0.0,2.0
"Toys ""R"" Us, Inc.",611,0,0.0,0.0,0.0,0.0,0.0,0.0
Celsius Network LLC,964,364,142.0,17284.0,63.0,5076.0,7495.0,6.0
FTX Trading Ltd.,1746,1018,197.0,73566.0,76.0,29061.0,27193.0,12.0
BlockFi Inc.,726,333,121.0,19995.0,84.0,8229.0,8758.0,6.0
Purdue Pharma L.P.,3066,1454,886.0,85468.0,87.0,21165.0,54902.0,1.0


### Spot Check
20 random clean entries per case. skim these after every run with new cases. if something looks off, figure out what general pattern it follows before adding a fix

In [44]:
for case, group in balanced.groupby("case_name"):
    print(f"\n### {case}")
    for _, r in group.sample(min(len(group), 20), random_state=0).iterrows():
        print(f"{r['date']} | {r['timekeeper']} | {r['hours']} | {r['task_codes']} | {r['narrative'][:150]}")


### BlockFi Inc.
02/09/23 | JOYNSON | 0.4 |  | CONDUCT REVIEW OF LOAN DOCUMENTS, COLLATERAL CALL DOCUMENTS, AND INVESTIGATIVE APPROACH DOCUMENTS
02/27/23 | AULET | 0.8 |  | REVIEW ISSUES RELATING TO LOAN CLAIMS AND DETERMINE ISSUES FOR FURTHER RESEARCH
03/09/23 | KRBECHEK | 0.3 |  | REVIEW AND REVISE RULE 2004 SUBPOENAS
1/25/23 | GSK | 0.1 | B110 | Review various Court notices.
03/17/23 | AULET | 4.2 |  | ATTEND D. SPACK DEPOSITION
08/15/23 | SILVERBERG | 1.1 |  | COMMITTEE CALL REGARDING EMERGENCE PLANNING, WALLET WITHDRAWAL ISSUE, PLANNING AND STRATEGY
03/06/23 | GILMAN | 0.8 |  | EVALUATE SEIZURE CLAIMS (.8); ANALYZE CLAIMS MADE IN ASSOCIATED INDICTMENT (1.3)
02/03/23 | SAWYER | 0.6 |  | INTERNAL BR MEETING RE CASE STATUS UPDATES, WORKSTREAM ALLOCATION, AND TIMELINE DISCUSSION
1/6/2023 | Meghji, Mohsin | 0.9 |  | Case Administration Attend Company/Committee Advisor Meeting
03/08/23 | AULET | 1.2 |  | FINANCE CALL WITH COMMITTEE
02/11/23 | BUSCARINO | 6.6 |  | TELEPHONE CONFERENCE W